## imports/etc

In [1]:
from bs4 import BeautifulSoup as beau
import pandas as pd
import itertools
import pickle
import nltk
import os
import re
import dask
from dask.distributed import Client, LocalCluster
client = Client(processes=True, n_workers=4, threads_per_worker=20)

2025-04-20 08:24:21,474 - distributed.scheduler - WARNING - Worker failed to heartbeat for 29842s; attempting restart: <WorkerState 'tcp://127.0.0.1:64714', name: 2, status: running, memory: 0, processing: 0>
2025-04-20 08:24:21,486 - distributed.scheduler - WARNING - Worker failed to heartbeat for 29842s; attempting restart: <WorkerState 'tcp://127.0.0.1:64715', name: 3, status: running, memory: 0, processing: 0>
2025-04-20 08:24:21,487 - distributed.scheduler - WARNING - Worker failed to heartbeat for 29842s; attempting restart: <WorkerState 'tcp://127.0.0.1:64716', name: 0, status: running, memory: 0, processing: 0>
2025-04-20 08:24:21,488 - distributed.scheduler - WARNING - Worker failed to heartbeat for 29842s; attempting restart: <WorkerState 'tcp://127.0.0.1:64717', name: 1, status: running, memory: 0, processing: 0>
2025-04-20 08:24:23,461 - distributed.nanny - WARNING - Restarting worker
2025-04-20 08:24:23,491 - distributed.nanny - WARNING - Restarting worker
2025-04-20 08:24

## set-up

In [2]:
fics_df = pd.DataFrame(columns=['L1', 'author', 'title', 'chapters', 'work', 'summary', 'notes', 'rating', 'warnings', 'fandoms', 'ships', 'characters', 'freeform', 'word_count'])
langs = ['Afrikaans', 'Albanian', 'Amharic', 'Anii', 'Arabic', 'Araona', 'Armenian', 'Assamese', 'Aymara', 'Ayoreo', 'Azerbaijan', 'Balanta', 'Bambara', 'Bariba', 'Basque', 'Bassari', 'Baure', 'Bedik', 'Belarusian', 'Bengali', 'Berber', 'Biali', 'Bislama', 'Boko', 'Bomu', 'Bosnian', 'Bozo', 'Buduma', 'Bulgarian', 'Burmese', 'Canichana', 'Cantonese', 'Carolinian', 'Catalan', 'Cayubaba', 'Chacobo', 'Chamorro', 'Chichewa', 'Chinese', 'Chirbawe', 'Comorian', 'Corsican', 'Creole', 'Croation', 'Czech', 'Dagaare', 'Dagbani', 'Dangme', 'Danish', 'Dari', 'Daroese', 'Dendi', 'Dhivehi', 'Dioula', 'Dogon', 'Dutch', 'Estonian', 'Fante', 'Figian', 'Filipino', 'Finnish', 'Foodo', 'Formosan', 'French', 'Fula', 'Gaelic', 'Gbe', 'Georgian', 'German', 'Gonja', 'Gourmanche', 'Greek', 'Guarani', 'Guarayu', 'Gujarati', 'Hakka', 'Hassaniya', 'Hausa', 'Hebrew', 'Hindi', 'Hiri', 'Hokkien', 'Hungarian', 'Icelandic', 'Igbo', 'Indonesian', 'Irish', 'Italian', 'Itene', 'Itonama', 'Japanese', 'Javanese', 'Jerriais', 'Jola', 'Kabye', 'Kalanga', 'Kallawaya', 'Kannada', 'Kanuri', 'Kasem', 'Kazakh', 'Khmer', 'Kinyarwanda', 'Kirundi', 'Kissi', 'Koisan', 'Korean', 'Kpelle', 'Kurdish', 'Kyrgyz', 'Lao', 'Latvian', 'Leco', 'Lithuanian', 'Lukpa', 'Luzembourgish', 'Macedonian', 'Malagasy', 'Malay', 'Malayalam', 'Malinke', 'Maltese', 'Mamara', 'Mandarin', 'Manding', 'Mandinka', 'Mandjak', 'Manipuri', 'Mankanya', 'Maori', 'Marathi', 'Marshallese', 'Mbelime', 'Meitei', 'Mongolian', 'Montenegrin', 'Moseten', 'Mossi', 'Motu', 'Movima', 'Moxos', 'Nambya', 'Nateni', 'Nauruan', 'Ndau', 'Ndebele', 'Nepali', 'Norwegian', 'Nzema', 'Oniyan', 'Oriya', 'Oromo', 'Ossetian', 'Pakawara', 'Palauan', 'Papiamento', 'Pashto', 'Persian', 'Pisin', 'Polish', 'Portuguese', 'Punjabi', 'Puquina', 'Quechua', 'Romanian', 'Romansh', 'Russian', 'Safen', 'Sango', 'Scots', 'Scottish', 'Sena', 'Serbian', 'Serer', 'Swedish', 'Shinhala', 'Shona', 'Siriono', 'Slovak', 'Slovene', 'Somali', 'Soninke', 'Sonsorolese', 'Sotho', 'Spanish', 'Susu', 'Swahili', 'Swati', 'Syenara', 'Tacana', 'Tagalog', 'Tajik', 'Tamasheq', 'Tamil', 'Tammari', 'Tapiete', 'Tasawaq', 'Tebu', 'Telugu', 'Tetum', 'Thai', 'Tigrinya', 'Tobian', 'Toma', 'Tonga', 'Tongan', 'Toromono', 'Tsonga', 'Tswana', 'Turkish', 'Turkmen', 'Tuvaluan', 'Twi', 'Ukrainian', 'Urdu', 'Uzbek', 'Venda', 'Vietnamese', 'Waama', 'Wamey', 'Weenhayek', 'Welsh', 'Wolof', 'Xhosa', 'Yaminawa', 'Yobe', 'Yom', 'Yoruba', 'Yuki', 'Yuracare', 'Zarma', 'Zulu']

In [3]:
def flat(lis):
    flatList = []
    # Iterate with outer list
    for element in lis:
        if type(element) is list:
            # Check if type is list then iterate through the sublist
            for item in element:
                flatList.append(item)
        else:
            flatList.append(element)
    return flatList

In [4]:
# USE SOUP
r_title = re.compile(r'(?<=<h2 class="title heading">).*?</', re.S)
r_rating = re.compile(r'Rating:</dt>\n<dd>.*?</dd>', re.S)
r_warnings = re.compile(r'Warnings*:</dt>\n<dd>.*?</dd>', re.S)
r_fandoms = re.compile(r'Fandoms*:</dt>\n<dd>.*?</dd>', re.S)
r_ships = re.compile(r'Relationships*:</dt>\n<dd>.*?</dd>', re.S)
r_characters = re.compile(r'Characters*:</dt>\n<dd>.*?</dd>', re.S)
r_freeform = re.compile(r'Additional Tags:</dt>\n<dd>.*?</dd>', re.S)

r_chapters = re.compile(r'(?<=Chapters:</dt><dd class="chapters">)\d*/\d*', re.S)
r_words = re.compile(r'(?<=Words:)\d[\d,]*', re.S)
#r_summary = re.compile(r'(?<=Summary).+', re.S)
#r_notes = re.compile(r'Notes.*?(?=\(See)', re.S)
# COME BACK TO MAKE SURE NOTES ISN'T GREEDY

#lone = re.compile(r'Work Text:.*<!--/chapter-->', re.S)
#lone_start = re.compile(r'Work Text:')
#solo = re.compile(r'(?<=<h3 class="landmark heading" id="work">Chapter Text</h3>).*', re.S)
#space = re.compile(r'\n\s*')
# GET BEAU.TEXT FROM HERE ON

In [5]:
def get_author(s):
    try:
        return s.find(rel='author').text
    except:
        return 'Anonymous'

In [6]:
def get_title(f):
    try:
        return beau(re.search(r_title, f)[0]).text
    except:
        return None

In [7]:
def get_words(f):
    try:
        x = re.search(r_words, f)[0]
        return convert_wc(re.sub(',', '', x))
    except:
        return re.search(r'(?<=Words:\n)\d*', beau(f).text)[0]
    
def convert_wc(x):
    try:
        return int(x)
    except:
        return None

In [8]:
def get_chapters(f):
    try:
        return re.search(r_chapters, f)[0]
    except:
        return None

In [9]:
notes = re.compile(r'notes module.*?</div>', re.S)
r_notes = re.compile(r'notes module">\n  <h3 class="heading">Notes:</h3>\n\n\n\n\n\n    <blockquote class="userstuff">\n      ', re.S)
r_notes2 = re.compile(r'notes module">\n        <h3 class="heading">Notes:</h3>\n          <blockquote class="userstuff">', re.S)
r_notes3 = re.compile(r'div class="notes module">\n  <h3 class="heading">Notes:</h3>\n\n\n\n\n\n\n    ', re.S)
r_notes4 = re.compile(r'notes module">\n  <h3 class="heading">Notes:</h3>\n  <blockquote class="userstuff">', re.S)
r_notes5 = re.compile(r'\(See the end of the work for (more )*notes\.\)', re.S)
r_notes6 = re.compile(r'notes module">\n  Notes:\n\n', re.S)

summary = re.compile(r'summary module.*?</div>', re.S)
r_summary = re.compile(r'summary module">\n          <h3 class="heading">Summary:</h3>\n            <blockquote class="userstuff">\n              ', re.S)
r_summary2 = re.compile(r'summary module">[\n\s]*Summary:', re.S)

work_a = re.compile(r'Work Text.*?</div>', re.S)
work_b = re.compile(r'Chapter Text.*?</div>', re.S)
r_work = re.compile(r'Chapter Text</h3>\n    ', re.S)
r_work2 = re.compile(r'Work Text:</h3>\n          <div class="userstuff">', re.S)
r_work3 = re.compile(r'Work Text:\n            \n\n\n\n\n\n\n\n', re.S)
# Work Text: 

In [10]:
ships = re.compile(r'Relationships*:.*?</dd>', re.S)
ships_scrub = re.compile(r'Relationships*:[\n\s]*(?=\S)', re.S)
charas = re.compile(r'Characters*:.*?</dd>', re.S)
charas_scrub = re.compile(r'Characters*:[\n\s]*(?=\S)', re.S)
fandoms = re.compile(r'Fandoms*:.*?</dd>', re.S)
fandoms_scrub = re.compile(r'Fandoms*:[\n\s]*(?=\S)', re.S)
warnings = re.compile(r'Warnings*.*?</dd>', re.S)
warnings_scrub = re.compile(r'Warnings*[\n\s]*(?=\<)', re.S)
warnings_fix = re.compile(r'Warning,:')
rating = re.compile(r'Rating:.*?</dd>', re.S)
rating_scrub = re.compile(r'Rating:[\n\s]*(?=\S)', re.S)
freeform = re.compile(r'Additional Tags*:.*?</dd>', re.S)
freeform_scrub = re.compile(r'Additional Tags*:[\n\s]*(?=\S)', re.S)

limk = re.compile(r'</')
space = re.compile(r'[\n\s]+', re.S)
trim = re.compile(r'\s[^\w\/\&\(]\s*,*', re.S)
fr_trim = re.compile(r'^, ')
trim_fr = re.compile(r',$')

def get_ships(f):
        try:
            f = re.sub(r',,', ', ', re.sub(ships_scrub, '', beau(re.sub(limk, ',</', re.search(ships, f)[0])).text))
            f = re.sub(trim_fr, '', re.sub(fr_trim, '', re.sub(trim, '', re.sub(space, ' ', f))))
            return f.split(', ')
        except:
           return None
def get_characters(f):
        try:
            f = re.sub(r',,', ', ', re.sub(charas_scrub, '', beau(re.sub(limk, ',</', re.search(charas, f)[0])).text))
            f = re.sub(trim_fr, '', re.sub(fr_trim, '', re.sub(trim, '', re.sub(space, ' ', f))))
            return f.split(', ')
        except:
           return None
def get_rating(f):
        try:
            f = re.sub(r',,', ', ', re.sub(rating_scrub, '', beau(re.sub(limk, ',</', re.search(rating, f)[0])).text))
            f = re.sub(trim_fr, '', re.sub(fr_trim, '', re.sub(trim, '', re.sub(space, ' ', f))))
            return f.split(', ')
        except:
           return None
def get_warnings(f):
        try:
            f = re.sub(r',,', ', ', (beau(re.sub(warnings_scrub, '', re.sub(limk, ',</', re.search(warnings, f)[0]))).text))
            f = re.sub(trim_fr, '', re.sub(fr_trim, '', re.sub(trim, '', re.sub(space, ' ', f))))
            f = re.sub(warnings_fix, '', f)
            return f.split(', ')
        except:
            return None
def get_fandoms(f):
        try:
            f = re.sub(r',,', ', ', re.sub(fandoms_scrub, '', beau(re.sub(limk, ',</', re.search(fandoms, f)[0])).text))
            f = re.sub(trim_fr, '', re.sub(fr_trim, '', re.sub(trim, '', re.sub(space, ' ', f))))
            return f.split(', ')
        except:
            return None
def get_freeform(f):
        try:
            f = re.sub(r',,', ', ', re.sub(freeform_scrub, '', beau(re.sub(limk, ',</', re.search(freeform, f)[0])).text))
            f = re.sub(trim_fr, '', re.sub(fr_trim, '', re.sub(trim, '', re.sub(space, ' ', f))))
            return f.split(', ')
        except:
            return None

In [11]:
def get_notes(f):
    return re.sub(r'notes module">[\n\s]*Notes:', '', ''.join([re.sub(r_notes6, '', re.sub(r_notes5, '', beau(re.sub(r_notes4, '', re.sub(r_notes3, '', re.sub(r_notes2, '', re.sub(r_notes, '', x))))).text)) for x in re.findall(notes, f)]))
def get_summary(f):
    return re.sub(r_summary2, '', ''.join([beau(re.sub(r_summary, '', x)).text for x in re.findall(summary, f)]))
def get_work(f):
    try:
        z = [beau(re.sub(r_work2, '', re.sub(r_work, '', x))).text for x in re.findall(work_a, f)]
        if z==[]:
            return [beau(re.sub(r_work2, '', re.sub(r_work, '', x))).text for x in re.findall(work_b, f)]
        else:
            return z
    except:
        return None

In [12]:
def parse_fic(f):
    s = beau(f)
    author = get_author(s)
    title = get_title(f)
    chapters = get_chapters(f)
    work = get_work(f)
    summary = get_summary(f)
    notes = get_notes(f)
    rating = get_rating(f)
    warnings = get_warnings(f)
    fandoms = get_fandoms(f)
    ships = get_ships(f)
    characters = get_characters(f)
    freeform = get_freeform(f)
    wordcount = get_words(s.text)
    return pd.DataFrame({'author':[author], 'title':[title], 'chapters':[chapters], 'work':[work], 'summary':[summary], 'notes':[notes], 'rating':[rating], 'warnings':[warnings], 'fandoms':[fandoms], 'ships':[ships], 'characters':[characters], 'freeform':[freeform], 'word_count':[wordcount]})

In [13]:
files = []
directory = 'natfinder/natfinder/scraped-works'
for f in os.scandir(directory):
    files.append(f.name)
for f in files:
    st = open(directory + '/' + f).read()
    temp_df = dask.delayed(parse_fic(st))
    fics_df = pd.concat([fics_df, temp_df.compute()], ignore_index=True)

In [14]:
fics_df['work'] = fics_df.work.map(lambda x : [re.sub(r'\s+', ' ', y).strip() for y in x])

In [15]:
fics_df['title'] = fics_df.title.map(lambda x : x.strip())

In [16]:
fics_df['punct'] = fics_df.work.map(lambda x : [z for z in set([y for y in ''.join(x)]) if not z.isalpha() and not z.isdigit()])

In [17]:
def find_lang(y):
    ls = []
    for l in langs:
        if l.lower() in y.lower():
            ls.append(l)
    return set(ls)

In [18]:
fics_df['L1_notes'] = fics_df.notes.map(lambda x : find_lang(''.join(x)))

In [19]:
fics_df['word_count'] = fics_df.word_count.map(lambda x : convert_wc(x))
fics_df['work'] = fics_df.work.map(lambda x : re.sub('Work Text:', '', ' '.join(x)))

## playground

In [20]:
def find_lang_tags(y):
    try:
        ls = []
        for l in langs:
            for x in y:
                if l.lower() in x.lower():
                    ls.append(l)
        return set(ls)
    except:
        return None
fics_df['L1_tags'] = fics_df.freeform.map(lambda x : find_lang_tags(x))

In [94]:
def validate(x):
    try:
        if len(x) != 0:
            return x
        else:
            return None
    except:
        return None
fics_df['L1_notes'] = fics_df.L1_notes.map(lambda x : validate(x))
fics_df['L1_tags'] = fics_df.L1_tags.map(lambda x : validate(x))

In [646]:
tags = fics_df.dropna(subset='L1_tags')
tags = tags.reset_index(drop=True)
tags = tags.drop(['chapters', 'work', 'rating', 'warnings', 'fandoms', 'ships', 'characters', 'freeform'], axis=1)

In [645]:
notes
pickle.dump(notes, open('scraped_notes.pkl', 'wb'))

In [665]:
notes = notes.drop(['summary', 'notes', 'word_count', 'punct', 'L1_tags', 'L1_notes'], axis=1)

In [669]:
fics = pd.merge(fics_df, notes, on=['author', 'title'], how="outer")
fics.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Data columns (total 18 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   L1_x        0 non-null      object
 1   author      380 non-null    object
 2   title       380 non-null    object
 3   chapters    379 non-null    object
 4   work        380 non-null    object
 5   summary     380 non-null    object
 6   notes       380 non-null    object
 7   rating      380 non-null    object
 8   warnings    380 non-null    object
 9   fandoms     380 non-null    object
 10  ships       341 non-null    object
 11  characters  368 non-null    object
 12  freeform    352 non-null    object
 13  word_count  380 non-null    int64 
 14  punct       380 non-null    object
 15  L1_notes    293 non-null    object
 16  L1_tags     60 non-null     object
 17  L1_y        205 non-null    object
dtypes: int64(1), object(17)
memory usage: 53.6+ KB


In [677]:
fics = fics.drop(['L1_x','L1_notes'],axis=1).reindex(columns=['L1_y', 'author', 'title', 'chapters', 'work', 'summary', 'notes', 'rating', 'warnings', 'ships', 'characters', 'freeform', 'word_count', 'punct', 'L1_tags'])

In [687]:
tags = fics.dropna(subset='L1_tags')

In [689]:
tags = tags.reset_index(drop=True)

In [789]:
tags = tags.drop(['L1_tags', 'punct', 'word_count', 'freeform', 'characters', 'ships', 'warnings', 'rating', 'notes', 'summary', 'work', 'chapters'], axis=1).dropna(subset='L1_y')

In [791]:
fics = pd.merge(fics, tags, on=['author', 'title'], how='outer')

In [796]:
fics = fics.rename(columns={'L1_y_x':'L1'})

In [805]:
fics['L1'] = fics['L1'].combine_first(fics['L1_y_y'])

In [ ]:
fics = fics.drop(['L1_tags', 'L1_y_y'], axis=1)

In [811]:
fics = fics.dropna(subset='L1')

In [ ]:
#pickle.dump(fics, open('scraped_fics_dataframe.pkl', 'wb'))

## dealing with punctuation

In [22]:
punctuation = set([y for y in ''.join([''.join(x) for x in fics_df['punct'].values])])

In [23]:
import unicodedata
import emoji
def is_emoji(s):
    return any(char in emoji.EMOJI_DATA for char in s)
for x in punctuation:
    if (not is_emoji(x)):
        for i, c in enumerate(x):  # insert your actual string
            print(f"Char {i}: '{c}' | Unicode: U+{ord(c):04X} | Name: {unicodedata.name(c, 'UNKNOWN')}")

Char 0: '♚' | Unicode: U+265A | Name: BLACK CHESS KING
Char 0: '/' | Unicode: U+002F | Name: SOLIDUS
Char 0: '^' | Unicode: U+005E | Name: CIRCUMFLEX ACCENT
Char 0: '！' | Unicode: U+FF01 | Name: FULLWIDTH EXCLAMATION MARK
Char 0: '(' | Unicode: U+0028 | Name: LEFT PARENTHESIS
Char 0: '₊' | Unicode: U+208A | Name: SUBSCRIPT PLUS SIGN
Char 0: '¡' | Unicode: U+00A1 | Name: INVERTED EXCLAMATION MARK
Char 0: ']' | Unicode: U+005D | Name: RIGHT SQUARE BRACKET
Char 0: '→' | Unicode: U+2192 | Name: RIGHTWARDS ARROW
Char 0: '、' | Unicode: U+3001 | Name: IDEOGRAPHIC COMMA
Char 0: '》' | Unicode: U+300B | Name: RIGHT DOUBLE ANGLE BRACKET
Char 0: '♔' | Unicode: U+2654 | Name: WHITE CHESS KING
Char 0: '。' | Unicode: U+3002 | Name: IDEOGRAPHIC FULL STOP
Char 0: ')' | Unicode: U+0029 | Name: RIGHT PARENTHESIS
Char 0: '݁' | Unicode: U+0741 | Name: SYRIAC QUSHSHAYA
Char 0: '<' | Unicode: U+003C | Name: LESS-THAN SIGN
Char 0: '॑' | Unicode: U+0951 | Name: DEVANAGARI STRESS SIGN UDATTA
Char 0: '¾' | Unico

In [24]:
punct_dict = {'"':'quotation mark', "'":'single quote', '“':'left smart quote', '”':'right smart quote', '‘':'left smart single quote', '’':'right smart single quote', '„':'low left quotation', '«':'left double angle bracket', '»':'right double angle bracket', '‹':'left single angle bracket', '›':'right single angle bracket', '〝':'reversed double ideographic', '〞':'double ideographic', '「':'left corner bracket', '」':'right corner bracket', '『':'left corner bracket bold', '』':'right corner bracket bold', '《':'left double-angle large', '》':'right double-angle large', '〈':'left single-angle large', '〉':'right single-angle large', '‚':'low left single'}

In [25]:
punct_list = list(punct_dict.keys())
len(punct_list)
## add em dash

22

In [26]:
punct_df = pd.DataFrame()
punct_df['L1'] = ''
for x in punct_list:
    punct_df[x] = ''
## if count(\s\?) > (\S\?)
## if count(\s!) > (\S!)
## dashes (--), (---), (\s-\s), (\S-\s) || (\s-\S), (en dash), (em dash), (\sen dash) || (en dash\s), (\sem dash), (em dash\s), etc
## periods, commas, exclamation marks, question marks, semicolon, dash, colon, parentheses


## other

In [27]:
#pickle.dump(df, open('fics_df_scraped_lang.pkl', 'wb'))

In [28]:
#def find_lang(fic):
    #try:
    #    for l in langs:
    #        if l.lower() in nltk.word_tokenize(fic.lower()):
    #            return l.upper()
    #except:
    #    return None
#def find_lang_tags(fic):
    #try:
    #    for l in langs:
    #        for x in fic:
    #            if l.lower() in nltk.word_tokenize(x.lower()):
    #                return l.upper()
    #except:
    #    return None

#fics_df['L1'] = fics_df.work.map(lambda x : dask.delayed(find_lang(x)))

In [ ]:
x = """
def rewrite(text, f):
    try:
        sfront = re.compile(r'(?<=<!-- END navigation -->).*', re.S)
        sback = re.compile(r'.*(?=<!-- END work -->)', re.S)
        sss = re.search(sfront, text)[0]
        ssss = re.search(sback, sss)[0]
        ssub = re.sub(r'</p><p>', '\n', ssss)
        with open('natfinder/natfinder/scraped-works/' + f, 'w') as file:
            file.write(ssub)
    except:
        return"""

In [30]:
x = """
def adlt(text, f):
    try:
        re.search(r'This work could have adult content. If you continue, you have agreed that you are willing to see such content.', text)[0]
        #os.remove(f)
    except:
        return False"""